# HW12 - Deep Reinforcement Learning
## Lunar Lander with Actor-Critic

演算法：Actor-Critic（REINFORCE with Baseline）
- Actor：輸出每個 action 的機率
- Critic：估計當前 state 的 value V(s)，作為 baseline
- Advantage：A_t = R_t - V(s_t)，降低 variance

## 安裝與環境設定

In [ ]:
# 安裝必要套件（Colab 環境）
!apt update -qq
!apt install python-opengl xvfb -y -qq
!pip install gym[box2d]==0.18.3 pyvirtualdisplay tqdm numpy==1.19.5 -q

In [ ]:
# 啟動虛擬顯示器（Colab 無法直接顯示 GUI）
from pyvirtualdisplay import Display
virtual_display = Display(visible=0, size=(1400, 900))
virtual_display.start()

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from IPython import display

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.distributions import Categorical
from tqdm.notebook import tqdm
import gym
import random

## 固定隨機種子

In [ ]:
seed = 543

def fix(env, seed):
    env.seed(seed)
    env.action_space.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

env = gym.make('LunarLander-v2')
fix(env, seed)

print('觀測空間維度:', env.observation_space.shape)  # (8,)
print('動作空間大小:', env.action_space.n)            # 4

## Actor-Critic 網路架構

共用特徵提取層（shared feature extractor），再分叉成：
- **Actor head**：輸出 4 個 action 的機率分佈
- **Critic head**：輸出純量 V(s)，代表目前 state 的預期總回報

In [ ]:
class ActorCriticNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        # 共用特徵提取層
        self.shared = nn.Sequential(
            nn.Linear(8, 64),
            nn.Tanh(),
            nn.Linear(64, 64),
            nn.Tanh(),
        )
        # Actor head：輸出 action 機率
        self.actor_head = nn.Linear(64, 4)
        # Critic head：輸出 state value V(s)
        self.critic_head = nn.Linear(64, 1)

    def forward(self, state):
        features = self.shared(state)
        action_probs = F.softmax(self.actor_head(features), dim=-1)
        state_value = self.critic_head(features)
        return action_probs, state_value

## Actor-Critic Agent

### 折扣累積 Reward 計算
對每個 time step t，計算：R_t = r_t + γ*r_{t+1} + γ²*r_{t+2} + ...

### Advantage 計算
A_t = R_t - V(s_t)，用來告訴 Actor「這個 action 比預期好還是差」

### 損失函數
- Actor loss = -log π(a_t|s_t) * A_t（最大化 advantage 加權的 log probability）
- Critic loss = (V(s_t) - R_t)²（讓 Critic 預測準確）
- Total loss = Actor loss + 0.5 * Critic loss

In [ ]:
class ActorCriticAgent:
    def __init__(self, network):
        self.network = network
        self.optimizer = optim.Adam(self.network.parameters(), lr=3e-4)
        self.gamma = 0.99  # 折扣因子

    def sample(self, state):
        """給定 state，從 Actor 抽樣 action，同時取得 Critic 的 V(s)"""
        state_tensor = torch.FloatTensor(state)
        action_probs, state_value = self.network(state_tensor)
        dist = Categorical(action_probs)
        action = dist.sample()
        log_prob = dist.log_prob(action)
        return action.item(), log_prob, state_value

    def discount_rewards(self, rewards):
        """計算每個 time step 的折扣累積 reward R_t"""
        discounted = []
        running = 0
        # 從最後一步往前計算
        for r in reversed(rewards):
            running = r + self.gamma * running
            discounted.insert(0, running)
        discounted = torch.FloatTensor(discounted)
        # 標準化，減少數值不穩定
        discounted = (discounted - discounted.mean()) / (discounted.std() + 1e-9)
        return discounted

    def learn(self, log_probs, state_values, rewards):
        """用一個 batch 的軌跡更新 Actor 和 Critic"""
        # 計算折扣累積 reward
        returns = self.discount_rewards(rewards)

        # 整理 tensor
        log_probs = torch.stack(log_probs)
        state_values = torch.stack(state_values).squeeze()

        # Advantage = R_t - V(s_t)，detach 讓 advantage 不參與 critic 的梯度計算
        advantages = returns - state_values.detach()

        # Actor loss：最大化 advantage 加權的 log prob
        actor_loss = (-log_probs * advantages).mean()

        # Critic loss：讓 V(s) 預測接近真實 R_t
        critic_loss = F.mse_loss(state_values, returns)

        # 合併 loss
        loss = actor_loss + 0.5 * critic_loss

        self.optimizer.zero_grad()
        loss.backward()
        # Gradient clipping，防止梯度爆炸
        nn.utils.clip_grad_norm_(self.network.parameters(), max_norm=0.5)
        self.optimizer.step()

    def save(self, path):
        torch.save({
            'network': self.network.state_dict(),
            'optimizer': self.optimizer.state_dict()
        }, path)

    def load(self, path):
        checkpoint = torch.load(path)
        self.network.load_state_dict(checkpoint['network'])
        self.optimizer.load_state_dict(checkpoint['optimizer'])

## 訓練迴圈

In [ ]:
network = ActorCriticNetwork()
agent = ActorCriticAgent(network)

EPISODE_PER_BATCH = 5   # 每個 batch 跑幾個 episode 再更新
NUM_BATCH = 500         # 總共跑幾個 batch

avg_total_rewards = []

agent.network.train()
prg_bar = tqdm(range(NUM_BATCH))

for batch in prg_bar:
    # 收集這個 batch 的所有軌跡資料
    all_log_probs = []
    all_state_values = []
    all_rewards = []
    total_rewards = []

    for episode in range(EPISODE_PER_BATCH):
        state = env.reset()
        total_reward = 0

        ep_log_probs = []
        ep_state_values = []
        ep_rewards = []

        while True:
            action, log_prob, state_value = agent.sample(state)
            next_state, reward, done, _ = env.step(action)

            ep_log_probs.append(log_prob)
            ep_state_values.append(state_value)
            ep_rewards.append(reward)

            state = next_state
            total_reward += reward

            if done:
                total_rewards.append(total_reward)
                break

        # 把這個 episode 的資料加進 batch
        all_log_probs.extend(ep_log_probs)
        all_state_values.extend(ep_state_values)
        all_rewards.extend(ep_rewards)

    # 用整個 batch 更新一次
    agent.learn(all_log_probs, all_state_values, all_rewards)

    avg_reward = sum(total_rewards) / len(total_rewards)
    avg_total_rewards.append(avg_reward)
    prg_bar.set_description(f"Avg Reward: {avg_reward:6.1f}")

# 儲存訓練好的模型
agent.save('actor_critic.pth')
print('訓練完成，模型已儲存。')

## 訓練曲線

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(avg_total_rewards, alpha=0.6, label='Avg Reward per Batch')

# 平滑曲線（每 20 個 batch 的移動平均）
window = 20
smoothed = np.convolve(avg_total_rewards, np.ones(window)/window, mode='valid')
plt.plot(range(window-1, len(avg_total_rewards)), smoothed, color='red', label=f'{window}-batch Moving Avg')

plt.xlabel('Batch')
plt.ylabel('Total Reward')
plt.title('Actor-Critic Training on LunarLander-v2')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'最後 50 個 batch 的平均 Reward: {np.mean(avg_total_rewards[-50:]):.1f}')

## 產生 Action List 供 JudgeBoi 評分

用訓練好的模型跑 5 個 episode，收集所有 action，存成 action_list.npy

In [ ]:
# 載入模型（如果剛訓練完可以跳過這步）
# agent.load('actor_critic.pth')

agent.network.eval()
fix(env, seed)

NUM_OF_TEST = 5  # JudgeBoi 要求跑 5 個 episode
action_list = []
total_scores = []

for episode in range(NUM_OF_TEST):
    state = env.reset()
    total_score = 0
    ep_actions = []

    while True:
        # eval 模式下取 argmax（greedy），不再隨機抽樣
        state_tensor = torch.FloatTensor(state)
        with torch.no_grad():
            action_probs, _ = agent.network(state_tensor)
        action = action_probs.argmax().item()

        ep_actions.append(action)
        state, reward, done, _ = env.step(action)
        total_score += reward

        if done:
            total_scores.append(total_score)
            break

    action_list.append(ep_actions)

# 輸出格式確認
print('Action list looks like:', action_list)
print('Action list\'s shape looks like:', np.shape(action_list))
print(f'5 個 episode 的平均分數: {np.mean(total_scores):.1f}')

In [ ]:
# 儲存 action list
np.save('action_list.npy', action_list)
print('action_list.npy 已儲存，可以上傳到 JudgeBoi。')

## 視覺化 Agent 表現（選用）

In [ ]:
agent.network.eval()
state = env.reset()
img = plt.imshow(env.render(mode='rgb_array'))

while True:
    state_tensor = torch.FloatTensor(state)
    with torch.no_grad():
        action_probs, _ = agent.network(state_tensor)
    action = action_probs.argmax().item()

    state, reward, done, _ = env.step(action)
    img.set_data(env.render(mode='rgb_array'))
    display.display(plt.gcf())
    display.clear_output(wait=True)

    if done:
        break